# Chest X-ray NORMAL/PNEUMONIA 최종 제출용 통합 노트북

이 노트북은 기존 두 개의 Python 스크립트를 Kaggle 제출용으로 통합한 버전입니다. 목표는 `NORMAL=0`, `PNEUMONIA=1` 이진 분류(binary classification)에서 384×384 입력 해상도, 3-backbone ensemble, 5-fold cross validation, OOF 기반 threshold optimization, pseudo-label fine-tuning, 최종 `submission.csv` 생성을 하나의 실행 흐름으로 정리하는 것입니다.

실제 의료 진단 목적이 아니라 연구/교육 및 대회 제출용 모델입니다. 리더보드 accuracy를 최적화하더라도 임상적 유효성을 의미하지 않으며, 실제 적용에는 external validation, 병원/촬영 조건 차이 검증, patient-level leakage 검토가 필요합니다.

## 1. 기존 두 코드의 역할

### `kaggle_ensemble_pipeline_384.py`

첫 번째 코드는 최종 모델의 **기본 학습 파이프라인(base ensemble training pipeline)** 역할입니다.

주요 기능은 다음과 같습니다.

- `train_split.csv`와 `val_split.csv`를 합친 뒤 `duplicate_group_id` 기준으로 `StratifiedGroupKFold` 5-fold 재분할
- 384×384 입력 이미지 사용
- CLAHE 전처리 + medically mild augmentation 적용
- DenseNet121 + CBAM, ConvNeXt-Tiny, EfficientNet-B3 총 3개 backbone 학습
- 각 backbone을 5-fold로 학습하여 총 15개 checkpoint 생성
- Smooth Focal Loss, Mixup, freeze → unfreeze fine-tuning, CosineAnnealingLR, 마지막 3 epoch checkpoint averaging 사용
- OOF prediction 기반으로 accuracy가 최대가 되는 decision threshold 탐색
- 4-way TTA로 최종 `submission.csv` 생성

### `pseudo_labeling_finetune_384.py`

두 번째 코드는 첫 번째 코드에서 만든 checkpoint를 활용하는 **pseudo-label adaptation pipeline** 역할입니다.

주요 기능은 다음과 같습니다.

- 저장된 15개 checkpoint 로드
- OOF validation F1을 다시 계산해 model ensemble weight 산출
- test set에 대해 ensemble probability 예측
- 확률이 높은 test sample만 pseudo-label로 선택
  - `prob >= 0.90` → PNEUMONIA pseudo-label
  - `prob <= 0.10` → NORMAL pseudo-label
- 각 fold의 원래 train fold에 pseudo-labeled test samples를 추가해 낮은 learning rate로 추가 fine-tuning
- pseudo-tuned checkpoint로 threshold 재탐색 및 최종 제출 파일 생성


In [ ]:
# =========================================================
# 0. Imports
# =========================================================
import os
import gc
import json
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedGroupKFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings("ignore")

## 2. Config

Kaggle에서 바로 실행하는 것을 기준으로 작성했습니다. 데이터셋 경로가 다르면 `DATA_ROOT`, `SAMPLE_SUBMISSION_PATH`만 수정하면 됩니다.

권장 실행 방식은 다음과 같습니다.

1. 처음 실행: `RUN_BASE_TRAINING=True`, `RUN_PSEUDO_LABELING=True`
2. 이미 checkpoint가 저장되어 있고 pseudo-label 단계만 다시 돌릴 때: `RUN_BASE_TRAINING=False`, `RUN_PSEUDO_LABELING=True`
3. pseudo-label 없이 기본 ensemble submission만 만들 때: `RUN_BASE_TRAINING=True`, `RUN_PSEUDO_LABELING=False`

주의: 384 해상도 × 3 models × 5 folds 구조이므로 Kaggle GPU 환경에서 실행 시간이 길 수 있습니다.

In [ ]:
# =========================================================
# 1. Configuration
# =========================================================
@dataclass
class CFG:
    # Kaggle paths
    data_root: str = "/kaggle/input/datasets/hyunwoo11/chest-xray-ai"
    work_dir: str = "/kaggle/working"
    sample_submission_path: str = "/kaggle/input/datasets/hyunwoo11/submission/sample_submission.csv"

    # Input files under data_root
    train_csv_name: str = "train_split.csv"
    val_csv_name: str = "val_split.csv"
    test_csv_name: str = "test.csv"
    image_dir_name: str = "data/images"

    # Run control
    run_base_training: bool = True
    run_pseudo_labeling: bool = True
    force_retrain: bool = False
    skip_existing_checkpoints: bool = True

    # Core hyperparameters
    seed: int = 42
    num_folds: int = 5
    image_size: int = 384
    epochs: int = 10
    epochs_ft: int = 3

    batch_size_dense: int = 16
    batch_size_conv: int = 16
    batch_size_eff: int = 16
    num_workers: int = 2

    lr_head: float = 1e-3
    lr_backbone: float = 1e-4
    lr_head_ft: float = 1e-4
    lr_backbone_ft: float = 1e-5
    weight_decay: float = 1e-4

    # Loss / augmentation
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0
    label_smoothing: float = 0.05
    mixup_alpha: float = 0.2
    mixup_prob: float = 0.5

    # Pseudo-label thresholds
    high_conf_pneumonia: float = 0.90
    high_conf_normal: float = 0.10

    # Inference threshold search
    threshold_min: float = 0.10
    threshold_max: float = 0.90
    threshold_step: float = 0.02

    # Runtime
    use_amp: bool = True
    fail_on_missing_image: bool = True

cfg = CFG()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DATA_ROOT = Path(cfg.data_root)
WORK_DIR = Path(cfg.work_dir)
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_ROOT / cfg.train_csv_name
VAL_CSV = DATA_ROOT / cfg.val_csv_name
TEST_CSV = DATA_ROOT / cfg.test_csv_name
IMAGE_DIR = DATA_ROOT / cfg.image_dir_name
SAMPLE_SUBMISSION_PATH = Path(cfg.sample_submission_path)

print("TRAIN_CSV:", TRAIN_CSV)
print("VAL_CSV:", VAL_CSV)
print("TEST_CSV:", TEST_CSV)
print("IMAGE_DIR:", IMAGE_DIR)

with open(WORK_DIR / "integrated_config_384.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2)

In [ ]:
# =========================================================
# 2. Reproducibility & Data Checks
# =========================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)


def validate_dataframe(df: pd.DataFrame, name: str, is_test: bool = False):
    required = ["file_name"] if is_test else ["file_name", "label"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")
    if not is_test:
        unique_labels = sorted(df["label"].dropna().unique().tolist())
        if not set(unique_labels).issubset({0, 1}):
            raise ValueError(f"{name} label must be binary 0/1. Found: {unique_labels}")


def get_group_column(df: pd.DataFrame) -> pd.Series:
    """Use duplicate_group_id when available; otherwise fall back to file_name groups."""
    if "duplicate_group_id" in df.columns:
        return df["duplicate_group_id"].astype(str)
    print("[WARN] duplicate_group_id column not found. Falling back to file_name as group key.")
    return df["file_name"].astype(str)

## 3. Dataset & Transform

입력 shape은 모든 모델에서 `[B, 3, 384, 384]`입니다. X-ray는 원본이 grayscale일 수 있지만 ImageNet pretrained backbone과 호환시키기 위해 RGB 3-channel로 변환합니다.

전처리는 `CLAHE(Contrast Limited Adaptive Histogram Equalization)`를 사용해 X-ray contrast를 보정합니다. Augmentation은 horizontal flip, mild rotation, brightness/contrast jitter 위주로 제한했습니다. Vertical flip은 흉부 X-ray 해부학적 구조를 뒤집기 때문에 사용하지 않습니다.

In [ ]:
# =========================================================
# 3. Dataset & Transforms
# =========================================================
class CLAHETransform:
    def __init__(self, clip_limit: float = 2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img_pil: Image.Image) -> Image.Image:
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_np = np.array(img_pil)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        return Image.fromarray(final_img)


class KaggleChestXrayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir: Path, transform=None, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, file_name: str) -> Path:
        candidates = [
            self.image_dir / file_name,
            self.image_dir / "train" / file_name,
            self.image_dir / "test" / file_name,
        ]
        for p in candidates:
            if p.exists():
                return p
        if cfg.fail_on_missing_image:
            raise FileNotFoundError(f"Image not found for {file_name}. Checked: {candidates}")
        return candidates[0]

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        file_name = row["file_name"]
        img_path = self._resolve_path(file_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            if cfg.fail_on_missing_image:
                raise
            image = Image.new("RGB", (cfg.image_size, cfg.image_size))

        if self.transform is not None:
            image = self.transform(image)

        if self.is_test:
            return image, file_name
        return image, int(row["label"])


train_transform = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((int(cfg.image_size * 1.1), int(cfg.image_size * 1.1))),
    transforms.CenterCrop(cfg.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# TTA scale A: 1.1x resize 후 center crop. 가장자리/마커 영향을 줄이는 margin-crop 관점.
val_test_transform_crop = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((int(cfg.image_size * 1.1), int(cfg.image_size * 1.1))),
    transforms.CenterCrop(cfg.image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# TTA scale B: 원본 전체 정보를 보존하는 direct full resize.
val_test_transform_full = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 4. Model Architecture

이 노트북은 서로 다른 inductive bias를 갖는 3개 backbone을 사용합니다.

- **DenseNet121 + CBAM**: dense connection으로 feature reuse가 강하고, CBAM을 추가해 channel/spatial attention을 부여합니다. 의료영상에서 미세한 texture와 폐 영역 attention 확인에 유리합니다.
- **ConvNeXt-Tiny**: modernized CNN 구조로 ResNet 계열보다 큰 receptive field와 안정적인 학습 특성을 기대할 수 있습니다.
- **EfficientNet-B3**: compound scaling 기반 CNN으로 384 해상도에서 성능 대비 효율이 좋습니다.

출력 shape은 세 모델 모두 `[B, 1]` logit입니다. 최종 확률은 `torch.sigmoid(logit)`으로 계산합니다.

In [ ]:
# =========================================================
# 4. Model Architectures
# =========================================================
class ChannelAttention(nn.Module):
    def __init__(self, in_planes: int, ratio: int = 16):
        super().__init__()
        hidden = max(in_planes // ratio, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, in_planes, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(x_cat))


class CBAM(nn.Module):
    def __init__(self, in_planes: int, ratio: int = 16, kernel_size: int = 7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_planes, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


class DenseNet121CBAM(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.cbam = CBAM(in_planes=1024)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1024, 1),
        )

    def forward(self, x):
        features = self.features(x)
        features = F.relu(features, inplace=True)
        features = self.cbam(features)
        out = F.adaptive_avg_pool2d(features, (1, 1))
        out = torch.flatten(out, 1)
        return self.classifier(out)


def get_densenet():
    return DenseNet121CBAM().to(DEVICE)


def get_convnext():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, 1)
    return model.to(DEVICE)


def get_efficientnet():
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    return model.to(DEVICE)


def build_model(model_name: str):
    if model_name == "DenseNet121":
        return get_densenet()
    if model_name == "ConvNeXt":
        return get_convnext()
    if model_name == "EfficientNet":
        return get_efficientnet()
    raise ValueError(f"Unknown model_name: {model_name}")


def get_param_groups(model: nn.Module, model_name: str):
    if model_name == "DenseNet121":
        backbone_params = model.features.parameters()
        head_params = list(model.classifier.parameters()) + list(model.cbam.parameters())
    elif model_name in ["ConvNeXt", "EfficientNet"]:
        backbone_params = model.features.parameters()
        head_params = model.classifier.parameters()
    else:
        raise ValueError(f"Unknown model_name: {model_name}")
    return backbone_params, head_params


def checkpoint_path(model_name: str, fold: int) -> Path:
    name_map = {
        "DenseNet121": "densenet121",
        "ConvNeXt": "convnext",
        "EfficientNet": "efficientnet",
    }
    return WORK_DIR / f"best_{name_map[model_name]}_384_fold{fold}.pt"

## 5. Loss, Mixup, Checkpoint Averaging

Loss는 `SmoothFocalLoss`입니다. `BCEWithLogitsLoss`에 focal weighting과 label smoothing을 결합한 형태입니다.

- 입력: model logits `[B, 1]`
- target: binary labels `[B, 1]`
- optimizer: `AdamW`
- scheduler: backbone unfreeze 후 `CosineAnnealingLR`
- metric: validation F1, OOF accuracy, AUROC, confusion matrix

리더보드 기준이 accuracy라면 최종 threshold는 F1이 아니라 OOF accuracy 최대 기준으로 찾는 것이 맞습니다. 다만 보고서에서는 의료영상 관점상 PNEUMONIA recall, NORMAL recall, F1, AUROC도 함께 해석해야 합니다.

In [ ]:
# =========================================================
# 5. Loss, Mixup, Checkpoint Averaging
# =========================================================
class SmoothFocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, smoothing: float = 0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        smoothed_targets = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        bce_loss = F.binary_cross_entropy_with_logits(inputs, smoothed_targets, reduction="none")
        pt = torch.exp(-bce_loss)
        alpha_factor = targets * self.alpha + (1.0 - targets) * (1.0 - self.alpha)
        focal_loss = alpha_factor * (1.0 - pt) ** self.gamma * bce_loss
        return torch.mean(focal_loss)


def mixup_data(x, y, alpha: float = 0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def average_checkpoints(checkpoint_paths):
    state_dicts = [torch.load(p, map_location="cpu") for p in checkpoint_paths]
    avg_state_dict = {}
    for key in state_dicts[0].keys():
        value = state_dicts[0][key]
        if torch.is_tensor(value):
            if value.is_floating_point():
                avg_state_dict[key] = torch.stack([sd[key] for sd in state_dicts]).mean(dim=0)
            else:
                avg_state_dict[key] = value
        else:
            avg_state_dict[key] = value
    return avg_state_dict


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# =========================================================
# 6. Fold Construction & Loaders
# =========================================================
def load_dataframes():
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)
    test_df = pd.read_csv(TEST_CSV)

    validate_dataframe(train_df, "train_df", is_test=False)
    validate_dataframe(val_df, "val_df", is_test=False)
    validate_dataframe(test_df, "test_df", is_test=True)

    full_df = pd.concat([train_df, val_df], ignore_index=True)
    full_df["label"] = full_df["label"].astype(int)

    print(f"train_df: {train_df.shape}")
    print(f"val_df:   {val_df.shape}")
    print(f"full_df:  {full_df.shape}")
    print(f"test_df:  {test_df.shape}")
    print("Class distribution in full_df:")
    print(full_df["label"].value_counts().sort_index())
    return train_df, val_df, full_df, test_df


def make_folds(full_df: pd.DataFrame):
    groups = get_group_column(full_df)
    sgkf = StratifiedGroupKFold(n_splits=cfg.num_folds, shuffle=True, random_state=cfg.seed)
    folds = list(sgkf.split(full_df, full_df["label"], groups=groups))
    print(f"Created {len(folds)} StratifiedGroupKFold splits.")
    return folds


def batch_size_for_model(model_name: str) -> int:
    if model_name == "DenseNet121":
        return cfg.batch_size_dense
    if model_name == "ConvNeXt":
        return cfg.batch_size_conv
    if model_name == "EfficientNet":
        return cfg.batch_size_eff
    raise ValueError(model_name)


def make_train_val_loaders(train_df, val_df, model_name: str):
    bs = batch_size_for_model(model_name)
    train_ds = KaggleChestXrayDataset(train_df, IMAGE_DIR, transform=train_transform, is_test=False)
    val_ds = KaggleChestXrayDataset(val_df, IMAGE_DIR, transform=val_test_transform_crop, is_test=False)
    train_loader = DataLoader(
        train_ds, batch_size=bs, shuffle=True, num_workers=cfg.num_workers,
        drop_last=True, pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=bs, shuffle=False, num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available()
    )
    return train_loader, val_loader


def make_test_loaders(test_df: pd.DataFrame, model_name: str):
    bs = batch_size_for_model(model_name)
    test_ds_crop = KaggleChestXrayDataset(test_df, IMAGE_DIR, transform=val_test_transform_crop, is_test=True)
    test_ds_full = KaggleChestXrayDataset(test_df, IMAGE_DIR, transform=val_test_transform_full, is_test=True)
    loader_crop = DataLoader(
        test_ds_crop, batch_size=bs, shuffle=False, num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available()
    )
    loader_full = DataLoader(
        test_ds_full, batch_size=bs, shuffle=False, num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available()
    )
    return loader_crop, loader_full

## 6. Base Training

각 fold에서 학습 흐름은 다음과 같습니다.

1. Epoch 1–2: backbone freeze, classifier head 중심 학습
2. Epoch 3: backbone unfreeze, 낮은 backbone learning rate로 warmup
3. Epoch 4 이후: backbone LR을 `1e-4`, head LR을 `1e-3`로 두고 cosine annealing 적용
4. 마지막 3 epoch checkpoint를 평균내어 SWA 유사 checkpoint 생성

검증 metric은 F1을 저장하지만, 최종 threshold는 이후 OOF accuracy 기준으로 별도 탐색합니다.

In [ ]:
# =========================================================
# 7. Training Engine
# =========================================================
def train_one_fold(model_name: str, fold: int, train_loader, val_loader):
    save_path = checkpoint_path(model_name, fold)

    if save_path.exists() and cfg.skip_existing_checkpoints and not cfg.force_retrain:
        print(f"[SKIP] Existing checkpoint found: {save_path}")
        val_f1 = evaluate_single_model_checkpoint(model_name, save_path, val_loader)["f1"]
        return save_path, val_f1

    print(f"\n>>> [Fold {fold+1}/{cfg.num_folds}] Training {model_name}")
    model = build_model(model_name)
    criterion = SmoothFocalLoss(cfg.focal_alpha, cfg.focal_gamma, cfg.label_smoothing)

    backbone_params, head_params = get_param_groups(model, model_name)
    for p in backbone_params:
        p.requires_grad = False
    optimizer = optim.AdamW(head_params, lr=cfg.lr_head, weight_decay=cfg.weight_decay)

    scheduler = None
    last_checkpoints = []
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))

    for epoch in range(cfg.epochs):
        if epoch == 2:
            print(">>> Unfreezing backbone with warmup LR")
            backbone_params, head_params = get_param_groups(model, model_name)
            for p in backbone_params:
                p.requires_grad = True
            optimizer = optim.AdamW([
                {"params": backbone_params, "lr": cfg.lr_backbone * 0.1},
                {"params": head_params, "lr": cfg.lr_head},
            ], weight_decay=cfg.weight_decay)

        elif epoch == 3:
            print(">>> Switching to full backbone LR with CosineAnnealingLR")
            backbone_params, head_params = get_param_groups(model, model_name)
            optimizer = optim.AdamW([
                {"params": backbone_params, "lr": cfg.lr_backbone},
                {"params": head_params, "lr": cfg.lr_head},
            ], weight_decay=cfg.weight_decay)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=max(cfg.epochs - 3, 1), eta_min=1e-6
            )

        model.train()
        running_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"{model_name} F{fold+1} E{epoch+1}/{cfg.epochs}", leave=False)
        for images, labels in pbar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float().unsqueeze(1)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                if random.random() < cfg.mixup_prob:
                    mixed_images, labels_a, labels_b, lam = mixup_data(images, labels, cfg.mixup_alpha)
                    outputs = model(mixed_images)
                    loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=running_loss / max(n_batches, 1))

        if scheduler is not None:
            scheduler.step()

        if epoch >= cfg.epochs - 3:
            temp_path = WORK_DIR / f"temp_{model_name.lower()}_384_fold{fold}_epoch{epoch}.pt"
            torch.save(model.state_dict(), temp_path)
            last_checkpoints.append(temp_path)

    print(f">>> Averaging last {len(last_checkpoints)} checkpoints for {model_name} fold {fold+1}")
    avg_state_dict = average_checkpoints(last_checkpoints)
    torch.save(avg_state_dict, save_path)
    model.load_state_dict(avg_state_dict)

    metrics = evaluate_model(model, val_loader)
    print(f"-> {model_name} Fold {fold+1} | Val Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f} | AUROC: {metrics['auroc']:.4f}")

    for p in last_checkpoints:
        if Path(p).exists():
            Path(p).unlink()

    del model, optimizer, scheduler, scaler
    cleanup_cuda()
    return save_path, metrics["f1"]


def train_base_ensemble(full_df: pd.DataFrame, folds):
    model_paths = {"DenseNet121": [], "ConvNeXt": [], "EfficientNet": []}
    model_f1s = {"DenseNet121": [], "ConvNeXt": [], "EfficientNet": []}

    for model_name in ["DenseNet121", "ConvNeXt", "EfficientNet"]:
        print(f"\n================ {model_name} 5-Fold Training ================")
        for fold, (train_idx, val_idx) in enumerate(folds):
            f_train_df = full_df.iloc[train_idx].reset_index(drop=True)
            f_val_df = full_df.iloc[val_idx].reset_index(drop=True)
            train_loader, val_loader = make_train_val_loaders(f_train_df, f_val_df, model_name)
            path, f1 = train_one_fold(model_name, fold, train_loader, val_loader)
            model_paths[model_name].append(path)
            model_f1s[model_name].append(f1)

    return model_paths, model_f1s

In [ ]:
# =========================================================
# 8. Evaluation & OOF Threshold Search
# =========================================================
def evaluate_model(model: nn.Module, loader, threshold: float = 0.5):
    model.eval()
    probs, labels_all = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            batch_probs = torch.sigmoid(outputs).detach().cpu().numpy().reshape(-1)
            probs.extend(batch_probs.tolist())
            labels_all.extend(labels.numpy().reshape(-1).tolist())

    probs = np.array(probs)
    labels_all = np.array(labels_all).astype(int)
    preds = (probs > threshold).astype(int)

    try:
        auroc = roc_auc_score(labels_all, probs)
    except ValueError:
        auroc = float("nan")

    return {
        "accuracy": accuracy_score(labels_all, preds),
        "f1": f1_score(labels_all, preds),
        "auroc": auroc,
        "probs": probs,
        "labels": labels_all,
        "preds": preds,
    }


def evaluate_single_model_checkpoint(model_name: str, ckpt_path: Path, val_loader):
    model = build_model(model_name)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    metrics = evaluate_model(model, val_loader)
    del model
    cleanup_cuda()
    return metrics


def load_model_paths_from_disk():
    model_paths = {
        "DenseNet121": [checkpoint_path("DenseNet121", f) for f in range(cfg.num_folds)],
        "ConvNeXt": [checkpoint_path("ConvNeXt", f) for f in range(cfg.num_folds)],
        "EfficientNet": [checkpoint_path("EfficientNet", f) for f in range(cfg.num_folds)],
    }
    missing = [str(p) for paths in model_paths.values() for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing checkpoints. Run base training first. Missing:\n" + "\n".join(missing))
    return model_paths


def evaluate_loaded_checkpoints(model_paths, full_df: pd.DataFrame, folds):
    print("\n[Evaluating loaded checkpoints to calculate ensemble weights]")
    f1s = {"DenseNet121": [], "ConvNeXt": [], "EfficientNet": []}

    for model_name in ["DenseNet121", "ConvNeXt", "EfficientNet"]:
        for fold, (_, val_idx) in enumerate(folds):
            val_df = full_df.iloc[val_idx].reset_index(drop=True)
            _, val_loader = make_train_val_loaders(val_df, val_df, model_name)
            metrics = evaluate_single_model_checkpoint(model_name, model_paths[model_name][fold], val_loader)
            f1s[model_name].append(metrics["f1"])
            print(f"{model_name} Fold {fold+1}: F1={metrics['f1']:.4f}")

    avg = {m: float(np.mean(v)) for m, v in f1s.items()}
    total = sum(avg.values()) + 1e-8
    weights = {m: avg[m] / total for m in avg}

    print("\nAverage F1:", avg)
    print("Ensemble weights:", weights)

    pd.DataFrame({
        "model": list(avg.keys()),
        "avg_f1": list(avg.values()),
        "ensemble_weight": [weights[m] for m in avg.keys()],
    }).to_csv(WORK_DIR / "ensemble_weights_384.csv", index=False)

    return weights, f1s


def predict_val_probs_single_model(model_name: str, ckpt_path: Path, val_loader):
    model = build_model(model_name)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    probs, labels_all = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            batch_probs = torch.sigmoid(outputs).detach().cpu().numpy().reshape(-1)
            probs.extend(batch_probs.tolist())
            labels_all.extend(labels.numpy().reshape(-1).tolist())

    del model
    cleanup_cuda()
    return np.array(probs), np.array(labels_all).astype(int)


def build_oof_predictions(model_paths, full_df: pd.DataFrame, folds, weights):
    print("\n[Building OOF ensemble predictions]")
    oof_probs = np.zeros(len(full_df), dtype=np.float32)
    oof_labels = np.zeros(len(full_df), dtype=np.int64)

    for fold, (_, val_idx) in enumerate(folds):
        val_df = full_df.iloc[val_idx].reset_index(drop=True)
        fold_probs = np.zeros(len(val_df), dtype=np.float32)
        fold_labels = None

        for model_name in ["DenseNet121", "ConvNeXt", "EfficientNet"]:
            _, val_loader = make_train_val_loaders(val_df, val_df, model_name)
            probs, labels = predict_val_probs_single_model(model_name, model_paths[model_name][fold], val_loader)
            fold_probs += weights[model_name] * probs.astype(np.float32)
            if fold_labels is None:
                fold_labels = labels

        oof_probs[val_idx] = fold_probs
        oof_labels[val_idx] = fold_labels

    return oof_probs, oof_labels


def find_best_threshold(oof_probs, oof_labels, prefix: str = "base"):
    best_threshold = 0.5
    best_acc = -1.0
    rows = []

    for threshold in np.arange(cfg.threshold_min, cfg.threshold_max + 1e-9, cfg.threshold_step):
        preds = (oof_probs > threshold).astype(int)
        acc = accuracy_score(oof_labels, preds)
        f1 = f1_score(oof_labels, preds)
        rows.append({"threshold": threshold, "accuracy": acc, "f1": f1})
        if acc > best_acc:
            best_acc = acc
            best_threshold = float(threshold)

    metric_df = pd.DataFrame(rows)
    metric_df.to_csv(WORK_DIR / f"threshold_search_{prefix}_384.csv", index=False)

    preds = (oof_probs > best_threshold).astype(int)
    try:
        auroc = roc_auc_score(oof_labels, oof_probs)
    except ValueError:
        auroc = float("nan")

    cm = confusion_matrix(oof_labels, preds)
    report = classification_report(oof_labels, preds, target_names=["NORMAL", "PNEUMONIA"], output_dict=True)

    summary = {
        "best_threshold": best_threshold,
        "accuracy": accuracy_score(oof_labels, preds),
        "f1": f1_score(oof_labels, preds),
        "auroc": auroc,
        "confusion_matrix": cm.tolist(),
        "classification_report": report,
    }

    with open(WORK_DIR / f"oof_metrics_{prefix}_384.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f">>> Best OOF Accuracy: {summary['accuracy']:.4f} at threshold={best_threshold:.3f}")
    print(f">>> OOF F1: {summary['f1']:.4f} | AUROC: {summary['auroc']:.4f}")
    print(">>> Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    return best_threshold, summary

## 7. Inference with 4-way TTA

각 checkpoint는 test image에 대해 4개의 view를 평균냅니다.

1. margin-crop 원본
2. margin-crop horizontal flip
3. full-resize 원본
4. full-resize horizontal flip

그 후 5 folds를 평균하고, 3개 모델은 validation 기반 weight로 weighted average합니다.

In [ ]:
# =========================================================
# 9. TTA Inference & Submission
# =========================================================
def predict_probs_tta(model: nn.Module, loader_crop, loader_full):
    model.eval()
    probs_list = []

    with torch.no_grad():
        for (images_crop, _), (images_full, _) in zip(loader_crop, loader_full):
            images_crop = images_crop.to(DEVICE, non_blocking=True)
            images_full = images_full.to(DEVICE, non_blocking=True)

            p_crop_orig = torch.sigmoid(model(images_crop))
            p_crop_flip = torch.sigmoid(model(torch.flip(images_crop, dims=[3])))
            p_full_orig = torch.sigmoid(model(images_full))
            p_full_flip = torch.sigmoid(model(torch.flip(images_full, dims=[3])))

            batch_probs = (p_crop_orig + p_crop_flip + p_full_orig + p_full_flip) / 4.0
            probs_list.extend(batch_probs.detach().cpu().numpy().reshape(-1).tolist())

    return np.array(probs_list, dtype=np.float32)


def predict_test_ensemble(model_paths, test_df: pd.DataFrame, weights):
    accumulated_probs = np.zeros(len(test_df), dtype=np.float32)

    for model_name in ["DenseNet121", "ConvNeXt", "EfficientNet"]:
        loader_crop, loader_full = make_test_loaders(test_df, model_name)
        model_weight = weights[model_name]

        for fold in range(cfg.num_folds):
            print(f"Predicting {model_name} fold {fold+1}/{cfg.num_folds}")
            model = build_model(model_name)
            model.load_state_dict(torch.load(model_paths[model_name][fold], map_location=DEVICE))
            fold_probs = predict_probs_tta(model, loader_crop, loader_full)
            accumulated_probs += model_weight * fold_probs / cfg.num_folds

            del model
            cleanup_cuda()

    return accumulated_probs


def save_submission(test_df: pd.DataFrame, probs: np.ndarray, threshold: float, filename: str = "submission.csv"):
    final_preds = (probs > threshold).astype(int)
    submission_df = pd.DataFrame({
        "file_name": test_df["file_name"].tolist(),
        "label": final_preds,
    })

    if SAMPLE_SUBMISSION_PATH.exists():
        sample_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        submission_df = sample_df[["file_name"]].merge(submission_df, on="file_name", how="left")
        if submission_df["label"].isna().any():
            raise ValueError("Some sample_submission file_name values were not predicted.")
        submission_df["label"] = submission_df["label"].astype(int)

    out_path = WORK_DIR / filename
    submission_df.to_csv(out_path, index=False)

    prob_path = WORK_DIR / filename.replace(".csv", "_probs.csv")
    pd.DataFrame({
        "file_name": test_df["file_name"].tolist(),
        "pred_prob": probs,
        "label": final_preds,
    }).to_csv(prob_path, index=False)

    print(f"Saved submission: {out_path}")
    print(f"Saved probabilities: {prob_path}")
    print("Prediction distribution:")
    print(submission_df["label"].value_counts().sort_index())
    return out_path, submission_df

## 8. Pseudo-label Fine-tuning

Pseudo-labeling은 test distribution에 적응시키는 semi-supervised trick입니다. 단, 불확실한 sample까지 넣으면 label noise가 커질 수 있으므로 high-confidence sample만 사용합니다.

이 노트북은 확률 기준을 보수적으로 설정합니다.

- `pred_prob >= 0.90`: PNEUMONIA로 pseudo-label
- `pred_prob <= 0.10`: NORMAL로 pseudo-label

Fine-tuning은 각 fold의 validation fold를 제외한 원래 training fold에 pseudo-labeled test samples를 추가해서 수행합니다. learning rate는 base training보다 낮게 설정해 기존 checkpoint를 크게 망가뜨리지 않도록 합니다.

In [ ]:
# =========================================================
# 10. Pseudo-label Extraction & Fine-tuning
# =========================================================
def extract_pseudo_labels(test_df: pd.DataFrame, test_probs: np.ndarray):
    pseudo_source = test_df.copy()
    pseudo_source["pred_prob"] = test_probs

    pneumonia_pseudo = pseudo_source[pseudo_source["pred_prob"] >= cfg.high_conf_pneumonia].copy()
    pneumonia_pseudo["label"] = 1

    normal_pseudo = pseudo_source[pseudo_source["pred_prob"] <= cfg.high_conf_normal].copy()
    normal_pseudo["label"] = 0

    pseudo_df = pd.concat([pneumonia_pseudo, normal_pseudo], ignore_index=True)
    pseudo_df["duplicate_group_id"] = [f"pseudo_{i}" for i in range(len(pseudo_df))]

    print("Pseudo-label extraction stats:")
    print(f"  total test samples: {len(test_df)}")
    print(f"  pneumonia pseudo:  {len(pneumonia_pseudo)}")
    print(f"  normal pseudo:     {len(normal_pseudo)}")
    print(f"  total pseudo:      {len(pseudo_df)}")
    print(f"  prob min/mean/median/max: {test_probs.min():.4f}/{test_probs.mean():.4f}/{np.median(test_probs):.4f}/{test_probs.max():.4f}")

    pseudo_save_path = WORK_DIR / "pseudo_labels_384.csv"
    pseudo_df[["file_name", "label", "duplicate_group_id", "pred_prob"]].to_csv(pseudo_save_path, index=False)
    print(f"Saved pseudo labels: {pseudo_save_path}")

    return pseudo_df[["file_name", "label", "duplicate_group_id"]]


def finetune_one_fold_on_pseudo(model_name: str, fold: int, ckpt_path: Path, train_loader):
    print(f"\n>>> [Fold {fold+1}/{cfg.num_folds}] Pseudo fine-tuning {model_name}")

    model = build_model(model_name)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    for p in model.parameters():
        p.requires_grad = True

    backbone_params, head_params = get_param_groups(model, model_name)
    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": cfg.lr_backbone_ft},
        {"params": head_params, "lr": cfg.lr_head_ft},
    ], weight_decay=cfg.weight_decay)
    criterion = SmoothFocalLoss(cfg.focal_alpha, cfg.focal_gamma, cfg.label_smoothing)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))

    for epoch in range(cfg.epochs_ft):
        model.train()
        running_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"Pseudo FT {model_name} F{fold+1} E{epoch+1}/{cfg.epochs_ft}", leave=False)
        for images, labels in pbar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float().unsqueeze(1)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=running_loss / max(n_batches, 1))

    torch.save(model.state_dict(), ckpt_path)
    print(f"-> Updated checkpoint: {ckpt_path}")

    del model, optimizer, scaler
    cleanup_cuda()


def run_pseudo_finetuning(model_paths, full_df: pd.DataFrame, folds, pseudo_df: pd.DataFrame):
    if len(pseudo_df) == 0:
        print("[WARN] No high-confidence pseudo-labels found. Skipping pseudo fine-tuning.")
        return model_paths

    for fold, (train_idx, _) in enumerate(folds):
        fold_train_df = full_df.iloc[train_idx].reset_index(drop=True)
        expanded_train_df = pd.concat([fold_train_df, pseudo_df], ignore_index=True)
        expanded_train_df["label"] = expanded_train_df["label"].astype(int)

        print(f"\nFold {fold+1}: original train={len(fold_train_df)}, pseudo={len(pseudo_df)}, expanded={len(expanded_train_df)}")

        for model_name in ["DenseNet121", "ConvNeXt", "EfficientNet"]:
            bs = batch_size_for_model(model_name)
            train_ds = KaggleChestXrayDataset(expanded_train_df, IMAGE_DIR, transform=train_transform, is_test=False)
            train_loader = DataLoader(
                train_ds, batch_size=bs, shuffle=True, num_workers=cfg.num_workers,
                drop_last=True, pin_memory=torch.cuda.is_available()
            )
            finetune_one_fold_on_pseudo(model_name, fold, model_paths[model_name][fold], train_loader)

    return model_paths

## 9. Main Execution

아래 셀 하나로 전체 흐름을 실행합니다.

최종 산출물은 기본적으로 다음 파일들입니다.

- `/kaggle/working/submission_base_384.csv`: pseudo-label fine-tuning 전 base ensemble 제출 파일
- `/kaggle/working/submission_384_pseudo.csv`: pseudo-label fine-tuning 후 제출 파일
- `/kaggle/working/submission.csv`: 최종 제출용 파일. pseudo-label을 사용하면 pseudo 결과가 저장되고, 사용하지 않으면 base 결과가 저장됩니다.
- `/kaggle/working/oof_metrics_base_384.json`, `/kaggle/working/oof_metrics_pseudo_384.json`: OOF 기반 성능 요약
- `/kaggle/working/threshold_search_*_384.csv`: threshold별 accuracy/F1 기록

In [ ]:
# =========================================================
# 11. Main Execution Flow
# =========================================================
def main():
    train_df, val_df, full_df, test_df = load_dataframes()
    folds = make_folds(full_df)

    print("\n================ Stage 1. Base Training / Checkpoint Loading ================")
    if cfg.run_base_training:
        model_paths, model_f1s = train_base_ensemble(full_df, folds)
    else:
        model_paths = load_model_paths_from_disk()
        model_f1s = None
        print("Loaded existing checkpoints from disk.")

    print("\n================ Stage 2. Base Ensemble Weight & OOF Threshold ================")
    weights, f1s = evaluate_loaded_checkpoints(model_paths, full_df, folds)
    oof_probs, oof_labels = build_oof_predictions(model_paths, full_df, folds, weights)
    base_threshold, base_summary = find_best_threshold(oof_probs, oof_labels, prefix="base")

    print("\n================ Stage 3. Base Submission ================")
    base_test_probs = predict_test_ensemble(model_paths, test_df, weights)
    save_submission(test_df, base_test_probs, base_threshold, filename="submission_base_384.csv")

    final_submission_name = "submission_base_384.csv"
    final_probs = base_test_probs
    final_threshold = base_threshold

    if cfg.run_pseudo_labeling:
        print("\n================ Stage 4. Pseudo-label Extraction ================")
        pseudo_df = extract_pseudo_labels(test_df, base_test_probs)

        if len(pseudo_df) > 0:
            print("\n================ Stage 5. Pseudo-label Fine-tuning ================")
            model_paths = run_pseudo_finetuning(model_paths, full_df, folds, pseudo_df)

            print("\n================ Stage 6. Pseudo-tuned Weight & Threshold ================")
            # Recalculate weights after pseudo fine-tuning.
            pseudo_weights, pseudo_f1s = evaluate_loaded_checkpoints(model_paths, full_df, folds)
            pseudo_oof_probs, pseudo_oof_labels = build_oof_predictions(model_paths, full_df, folds, pseudo_weights)
            pseudo_threshold, pseudo_summary = find_best_threshold(pseudo_oof_probs, pseudo_oof_labels, prefix="pseudo")

            print("\n================ Stage 7. Pseudo-tuned Submission ================")
            pseudo_test_probs = predict_test_ensemble(model_paths, test_df, pseudo_weights)
            save_submission(test_df, pseudo_test_probs, pseudo_threshold, filename="submission_384_pseudo.csv")

            final_submission_name = "submission_384_pseudo.csv"
            final_probs = pseudo_test_probs
            final_threshold = pseudo_threshold
        else:
            print("No pseudo-label fine-tuning was performed. Base submission remains final.")

    print("\n================ Stage 8. Save Final submission.csv ================")
    save_submission(test_df, final_probs, final_threshold, filename="submission.csv")
    print(f"\nDone. Final submission source: {final_submission_name}")


main()

## 10. 제출 전 확인 포인트

1. `/kaggle/working/submission.csv`가 생성되었는지 확인합니다.
2. `label` 값이 0 또는 1만 있는지 확인합니다.
3. `sample_submission.csv`의 `file_name` 순서와 정확히 align되었는지 확인합니다.
4. pseudo-labeling을 사용했다면 대회 규칙상 test pseudo-label fine-tuning이 허용되는지 확인합니다.
5. validation/OFF 성능이 좋아도 public/private leaderboard와 distribution 차이가 있을 수 있으므로 한 번의 점수 변화만으로 과도하게 결론 내리지 않습니다.

In [ ]:
# Optional sanity check after running main()
submission_path = WORK_DIR / "submission.csv"
if submission_path.exists():
    sub = pd.read_csv(submission_path)
    print(sub.head())
    print(sub["label"].value_counts().sort_index())
    assert set(sub["label"].unique()).issubset({0, 1})
    print(f"Ready to submit: {submission_path}")
else:
    print("submission.csv has not been created yet. Run the main execution cell first.")